In [1]:
import os
import pandas as pd
import numpy as np

# 1. 원본 데이터 로드 (상대 경로)
raw_path = '../data/raw/all-weeks-global.tsv'
df_raw = pd.read_csv(raw_path, sep='\t')

print(f"1. 원본 전체 데이터 크기: {df_raw.shape[0]:,}행, {df_raw.shape[1]}개 열")

# 2. 비영어 TV 카테고리 필터링 및 날짜 변환
df_non_eng = df_raw[df_raw['category'] == 'TV (Non-English)'].copy()
df_non_eng['week'] = pd.to_datetime(df_non_eng['week'])
df_non_eng = df_non_eng.sort_values(by=['week', 'weekly_rank']).reset_index(drop=True)

print(f"2. 비영어 TV 시리즈 필터링 완료: 총 {len(df_non_eng):,}개 행")
print(f"3. 수집 주차(Week) 수: 총 {df_non_eng['week'].nunique()}주 (약 5년치)")
print(f"4. 데이터 기간: {df_non_eng['week'].min().strftime('%Y년 %m월 %d일')} ~ {df_non_eng['week'].max().strftime('%Y년 %m월 %d일')}")

# 3. 1차 정제본 저장 (interim)
interim_path = '../data/interim/non_english_tv_clean.csv'
df_non_eng.to_csv(interim_path, index=False)
print(f"5. 1차 정제 파일 저장 완료 -> {interim_path}")

# 4. 상위 3개 행 미리보기 (한글 컬럼명 및 시청 시간 단위 포맷 적용)
preview_df = df_non_eng[['week', 'weekly_rank', 'show_title', 'weekly_hours_viewed', 'cumulative_weeks_in_top_10']].head(3).copy()

# 날짜 포맷 정리 (YYYY-MM-DD)
preview_df['week'] = preview_df['week'].dt.strftime('%Y-%m-%d')

# 시청 시간 읽기 편하게 콤마 표기
preview_df['weekly_hours_viewed'] = preview_df['weekly_hours_viewed'].apply(lambda x: f"{x:,.0f} 시간")

# 컬럼명 한글로 변경
preview_df = preview_df.rename(columns={
    'week': '집계 주차',
    'weekly_rank': '주간 순위',
    'show_title': '작품명',
    'weekly_hours_viewed': '주간 시청 시간',
    'cumulative_weeks_in_top_10': 'Top 10 누적 진입(주)'
})

preview_df

1. 원본 전체 데이터 크기: 10,960행, 9개 열
2. 비영어 TV 시리즈 필터링 완료: 총 2,740개 행
3. 수집 주차(Week) 수: 총 274주 (약 5년치)
4. 데이터 기간: 2021년 07월 04일 ~ 2026년 09월 27일
5. 1차 정제 파일 저장 완료 -> ../data/interim/non_english_tv_clean.csv


,집계 주차,주간 순위,작품명,주간 시청 시간,Top 10 누적 진입(주)
0,2021-07-04,1,Elite,"38,720,000 시간",1
1,2021-07-04,2,The Seven Deadly Sins,"33,240,000 시간",1
2,2021-07-04,3,"Newly Rich, Newly Poor","28,270,000 시간",1
